### 2층 신경망 구성하여 오차 역전파를 수동으로 구현해보기
- 은닉층 포함된 MLP에서 오차역전파 흐름 전체를 자동 미분 없이 수동미분으로 구현한다.
- forward (순전파) -> loss (손실 계산) -> backward (역전파) -> gradient(기울기) 업데이트  

In [10]:
import torch

x = torch.tensor([[0.5, 0.8]])      # 1,2
y_true = torch.tensor([[1.0]])      # 1,1

print(f"x.size = {x.size()}, y_treu.size = {y_true.size()}")

x.size = torch.Size([1, 2]), y_treu.size = torch.Size([1, 1])


In [11]:
# 가중치/편향 초기화
W1 = torch.tensor([[0.1, 0.3],
                  [0.2, 0.4]], requires_grad=False) # 입력(2) -> 은닉(2)로 각각 가중치 행렬
b1 = torch.tensor([[0.1, 0.2]], requires_grad=False) # 은닉층 편향 벡터

print(W1)
print(b1)

tensor([[0.1000, 0.3000],
        [0.2000, 0.4000]])
tensor([[0.1000, 0.2000]])


requires_grad=False : Pytorch가 .backward() 시 자동으로 기울기를 계산하지 않음

In [12]:
# 가중치/편향 초기화
W2 = torch.tensor([[0.5],
                  [0.6]], requires_grad=False) # 은닉(2) -> 출력(1) 가중치 행렬 (2,1)
b2 = torch.tensor([[0.3]], requires_grad=False) # 출력층 편향 스칼라 (1,1). 출력노드가 1개이므로 편향도 1개

print(W2)
print(b2)

tensor([[0.5000],
        [0.6000]])
tensor([[0.3000]])


In [13]:
# 시그모이드 활성화 함수 : 입력을 0~1로 압축해서 반환 (비선형성 부여)
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))  # 지수계산 후 시그모이드 공식 적용

# 시그모이드 미분 함수 : 역전파에서 기울기 계산에 사용
def sigmoid_deriv(x):
    s = sigmoid(x)                  # 순전파 시그모이드 출력값 s를 미분 계산
    return s * (1-s)                # sigmoid'(x) = s(1-s) => 기울기 계산

In [15]:
z1 = x @ W1 + b1        # 첫 번째 은닉층 선형 결합 : (1,2) @ (2, 2) + (1, 2) = shape : (1, 2)
a1 = sigmoid(z1)        # 첫 번째 은닉층 활성화함수  

z2 = a1 @ W2 + b2       # 출력층 선형결합 : (1, 2)  @ (2, 1) + (1, 1) = shape : ()
a2 = sigmoid(z2)        # 출력층 활성화함수 : 0~1 사이값으로 반환

print(z1)
print(a1)
print(z2)
print(a2)

tensor([[0.3100, 0.6700]])
tensor([[0.5769, 0.6615]])
tensor([[0.9853]])
tensor([[0.7282]])


@ 연산자를 사용
텐서끼리 연산할 시 행렬곱 수행
(1,2) @ (2,2) => (1,2)


In [17]:
# 역전파
# MSE 평균제곱오차로 손실 계산 (0.5를 곱한건 미분시 2가 상쇄되어 깔끔하게 약분된다. 기울기계산 단순화)
loss = 0.5 * (y_true - a2) ** 2

In [ ]:
# 기울기 계산
delta2 = (a2 - y_true) * sigmoid_deriv(z2)

dW2 = a1.T @ delta2